# Long Baseline Study of Mira Variable Stars: Part Three
## 24-in Observations Calibrated with a Standard Star
The majority of this notebook is taken from Emily's Aperture Photometry notebook, with adjustments made to calculate the magnitude in the end. Check each cell for multiple hashtags that indicate where you need to input your own info.

#### Step One
Use Emily's Image Calibration notebook to sort and calibrate your observation files.

#### Step Two
Read in your first observation file.

In [ ]:
#Start by importing our packages!
%matplotlib widget

# for handling arrays; image data are arrays!
import numpy as np 
# for plotting and displaying data
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec
# for making lists of filenames
from glob import glob 
# for reading and writing FITS format data
from astropy.io import fits 
# for doing aperture photometry:
import photutils
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, CircularAnnulus
from astropy.stats import mad_std
# Import a function for fitting data
from scipy.optimize import curve_fit
# need to manually trigger garbage collector to free memory
import gc
from IPython.display import display, clear_output
# need for filepath and dataframe
import os
import pandas as pd
from datetime import datetime
from astropy.time import Time
import matplotlib.dates as mdates
from astropy.table import Table
from matplotlib.ticker import MultipleLocator


############### Set the platescale for your observation ##################

#platescale = 0.36 # arcsec / pixel    #ProEM Camera on the McDonald 82" telescope
#platescale = 0.964016 # arcsec / pixel   #CMOS on the Yerkes 41" telescope
platescale = 0.6 # arcsec / pixel    #CCD on the Yekres 24" telescope

#platescale=        # place to put platescales for other telescopes

In [ ]:
# Read in your *calibrated* image file

################## Filename of example frame relative to your current directory ########################
firstfilename = '/raid6/users/rantoine/2026-07-14/lights/7-14-26/7-14-26_c/7-14-26_KZ_Her_-001_240s_V_c.fits'

# load the data
firstframe = fits.getdata(firstfilename)

# display the image
plt.figure(1) #Create new "figure" for this image
plt.imshow(firstframe,cmap='gray',vmin=np.percentile(firstframe,25),
           vmax=np.percentile(firstframe,99), origin = 'lower')

plt.show()

In [ ]:
plt.close()
# First thing we need to know is how much scatter is in the "background"
# counts so that we don't detect random background fluctuations as stars.
bkg_sigma = mad_std(firstframe)

# We also need to guess the full-width at half maximum (FWHM) for the 
# stars to guide our search, which we will verify in a later step.

######### change fwhm value to the estimate you have from your observation ##########

daofind = DAOStarFinder(fwhm=4, threshold=5*bkg_sigma)
sources = daofind(firstframe) 
sources #display results

#### Step Three
Start performing aperture photometry for you target star.

In [ ]:
# Using the widget tool, hover your mouse over the display of the first frame and determine the pixel coordinates of 
# your target star and your comparison star. We will provide these to DAOStarFinder as estimate coordinates 
# We will check for each frame which detected "star" is nearest to these 
# coordinates to validate the detections.

######## Set the pixel coordinates for your target star and your comparison star ##########

# Target coordinates:
tx,ty = (657,715)

# For your comparison star, pick a star of relatively similar size to your target star, but note that this will not be the final comparison star used to calculate magnitude.
# Comparison star coordinates:
cx,cy = (490,492)

# Define function that returns the distance, in pixels, between two points:
def distance(x1,y1,x2,y2):
    return np.sqrt((x1-x2)**2.+(y1-y2)**2.)

# Set up empty arrays to hold distances of detections from approximate locations
numsources = len(sources)
tdist = np.zeros(numsources)
cdist = np.zeros(numsources)

# Populate arrays with distances for each source
for i,row in enumerate(sources):
    tdist[i] = distance(tx,ty,row["x_centroid"],row["y_centroid"])
    cdist[i] = distance(cx,cy,row["x_centroid"],row["y_centroid"])
    
# Identify the row indices of the target and comp stars in "sources"
tind = np.argmin(tdist)
cind = np.argmin(cdist)

print('Target index: {}. Comparison index: {}. '.format(tind,cind))

# Replace "sources" with just target and comparison, in that order:
sources = sources[[tind,cind]]

In [ ]:
# Let's show where those detections are on the image...

plt.figure(num=2) #Create new "figure" for this image
plt.imshow(firstframe,cmap='gray',vmin=np.percentile(firstframe,5),
           vmax=np.percentile(firstframe,98), origin = 'lower')
plt.scatter(sources['x_centroid'],sources['y_centroid'],c=['blue','red'],
            marker='x')
plt.show()

In [ ]:
plt.close()
# Visualize the target star profile.

# Define the Gaussian function for fitting the stellar profile
def gaussian(x, A, sigma):
    # Definition of a Gaussian where:
    # - x is distance from center
    # - A is amplitude
    # - sigma is stddev
    # (FWHM = 2.3548*sigma)
    return A*np.exp(-(x)**2/(2.*sigma**2))

# Plot the pixel values in terms of distance from the
# center of the first star for pixels within ~15 of the pixels

x0 = sources['x_centroid'][0] # x center of target star
y0 = sources['y_centroid'][0] # y center of target star
pdist = 15 #distance in pixels to examine
dist = [] #empty list to hold pixel distances
pval = [] #empty list to hold pixel values

for x in np.arange(round(x0)-pdist,round(x0)+pdist):
    for y in np.arange(round(y0)-pdist,round(y0)+pdist):
        dist.append(distance(x,y,x0,y0))
        # It bothers me that the image is indexed as (y,x)
        pval.append(firstframe[int(y),int(x)])

# Plot the results (in units of pixels) in a new figure
fig, ax1 = plt.subplots(num=3, sharey='row')
ax1.scatter(np.array(dist),pval,c='gray')
ax1.set_xlim(0,pdist)
ax1.set_xlabel('distance (pixels)')
ax1.set_ylabel('pixel values')

#Make a second x-axis with arcsec units
ax2 = ax1.twiny()
ax2.set_xlabel('distance (arcsec)')
ax2.set_xlim(0,pdist*platescale)

# Find a Gaussian curve of best fit to the observed stellar
# profile above the median image background (the scipy.optimize 
# functions are great for fitting data!)

# "Typical background"
bkgmed = np.median(firstframe.flatten())

########################### Set initial guess for peak pixel value and sigma ##################################

p0=[20000.,1.] #initial guesses of height and sigma
popt,_  = curve_fit(gaussian,dist,np.array(pval)-bkgmed,p0=p0)
fwhm = np.abs(popt[-1])*2.3548*platescale
print("Seeing in the first frame is "+str(fwhm)+'"')

# Plot this best fit
gsamplespacing = 0.1
gsample=np.arange(0,pdist+gsamplespacing,gsamplespacing)
ax2.plot(gsample*platescale,bkgmed+
         gaussian(gsample,popt[0],popt[1]),c='r',lw=3)
# And show the HWHM
ax2.plot([0,fwhm/2.,fwhm/2.],[bkgmed+(popt[0]/2.),bkgmed+(popt[0]/2.),bkgmed],
         c='black',ls='-',lw=2)
ax2.annotate('FWHM: {:.2f}"'.format(fwhm), xy=(fwhm/2.,bkgmed+popt[0]/2.), 
             xycoords='data',xytext=(0.57,0.5), textcoords='axes fraction',
             arrowprops=dict(facecolor='black', shrink=0.05,width=1),
             ha='center',va='center',fontsize=18)

# Set y-limits to show gaussian fit well
ymax = popt[0]*1.1
ax2.set_ylim(0,ymax)
ax1.set_ylim(0,ymax)

# Show target apertures and annulus on this plot.

######################## Set aperture sizes you want to test along with sky annuli ####################
   
apsizes = np.arange(2,14,0.5) 
print("Aperture sizes (in pixels) to test:",apsizes)
for apsize in apsizes:
    ax1.plot([apsize,apsize],[0,ymax],'b--')

######################## Set annulus region you want to test to capture sky background ####################
     
ax1.fill_between([11,14],[ymax,ymax],color='g',edgecolor='g',hatch='x',zorder=-1,alpha=0.3)

plt.show()

In [ ]:
plt.close()
# So we want to perform aperture photometry around the detected star centers
# Let's first show what an example aperture looks like:

# display the image
plt.figure(4) #Create new "figure" for this image
# Use different vmin/vmax to get better contrast
plt.imshow(firstframe,cmap='gray',vmin=np.percentile(firstframe,25),
           vmax=np.percentile(firstframe,99.9))

# Put star centroid positions in format for aperture photometry
positions = [(x,y) for x,y in sources['x_centroid','y_centroid']]

# Define an example circular aperture of radius 
ap = CircularAperture(positions,r=7)
ap.plot(color='blue',linestyle='--',lw=1)

# Define sky annulus in range 3 pixel radius
skyan = CircularAnnulus(positions,r_in=24,r_out=28)
skyan.plot(color='green',hatch='x',lw=1)

# Set limits to plot only around one star for demonstration
plt.xlim(x0-50,x0+50)
plt.ylim(y0+50,y0-50)
plt.show()

In [ ]:
plt.close()
# Sum the pixel values in each aperture (target and comparison) and sky annuli
starapsum = ap.do_photometry(firstframe)[0]
print("Aperture sums: ",starapsum)

skysum = skyan.do_photometry(firstframe)[0]
print("Sky sums: ",skysum)

# But we don't just want the total counts in each aperture...
# We want the average sky counts from the sky annuli which we will use to
# subtract the sky counts from the star aperture counts.

# The area() call returns the aperture areas, which we can use to correct
# for the background.
averagesky = skysum/skyan.area

print("Average sky counts: ",averagesky)

# Then use this average to subtract the expected contribution from the sky
# to the star apertures:

starsum = starapsum - ap.area*averagesky

print("\nStar sums in aperture: \nTarget {} \nComp 1 {} ".format(starsum[0],starsum[1]))

# Finally, we want to divide counts of the target star by the counts from
# the comparison star to correct for changes in observing conditions.
rel_flux1 = starsum[0] / starsum[1]
print("\nRelative flux of target w Comp 1: ",rel_flux1)

In [ ]:
#Now we are using all of the observations.
# Get list of reduced filenames

###### change the file name ######

filenames = sorted(glob('/raid6/users/rantoine/2026-07-24/lights/7-24-26/7-24-26_c/7-24-26_R_Aql*_c.fits')) #use asterik to fill in the rest and leave out other objects
n_frames = len(filenames)
print(n_frames)


# Verify from the first few filenames that these are the reduced
# frames and are listed in order.
print(filenames[:4])

# Remember the aperture sizes we will test:
print("Aperture sizes (in pixels) to test:",apsizes)
n_apers = apsizes.shape[0]

# And we have 2 stars that we are measuring:
n_stars = 2

# To hold all of our results, we set up a 3-dimensional numpy array
# with dimensions (n_apers x n_stars x n_frames):
photresults = np.zeros((n_apers,n_stars,n_frames))

print("Photometry results will have shape",photresults.shape)

# You need to index this array in the same order, so the first index
# selects the aperture size, the next index selects the star (target
# at index 0; comp star at index 1), and frame number is last.

In [ ]:
# We want to show a lot of plots together as we do the photometry.
# Set those up first.
%matplotlib inline
# Visualize frames as we do photometry

fig = plt.figure()
ax1 = fig.add_subplot(1, 1, 1) 

# We'll also set up empty lists to hold seeing and sky brightness
skybrightness = []
fwhm = []

for i,filename in enumerate(filenames):
#for i,filename in enumerate(filenames[990:1000]):
    # load the data
    frame = fits.getdata(filename)

    # First plot (upper left) displays images with apertures/annulli
    ax1 = plt.subplot()
    ax1.set_xlim(0, frame.shape[0])
    ax1.set_xlabel("x pixel")
    ax1.set_ylim(0, frame.shape[1])
    ax1.set_ylabel("y pixel")
    ax1.set_title("Image {} / {}".format(0,n_frames))
    showframe = ax1.imshow(np.zeros((frame.shape[0],frame.shape[1])),cmap='gray')
    
    # Set up circles to indicate apertures used
    targetap = plt.Circle((0, 0), 9, color='b',fill=False,lw=2)
    targetan1 = plt.Circle((0, 0), 18, color='g',fill=False,lw=2)
    targetan2 = plt.Circle((0, 0), 35, color='g',fill=False,lw=2)
    compap = plt.Circle((0, 0), 9, color='r',fill=False,lw=2)
    compan1 = plt.Circle((0, 0), 18, color='g',fill=False,lw=2)
    compan2 = plt.Circle((0, 0), 35, color='g',fill=False,lw=2)
    ax1.add_artist(targetap)
    ax1.add_artist(targetan1)
    ax1.add_artist(targetan2)
    ax1.add_artist(compap)
    ax1.add_artist(compan1)
    ax1.add_artist(compan2)
    
    # display the image
    showframe.set_data(frame)
    showframe.set_clim(vmin=np.percentile(frame,5),vmax=np.percentile(frame,98))
    ax1.imshow(frame,cmap='gray',vmin=np.percentile(frame,5),vmax=np.percentile(frame,98))
    ax1.set_title("Image {} / {}".format(i+1,n_frames))
    
    # First thing we need to know is how much scatter is in the "background"
    # counts so that we don't detect random background fluctuations as stars.
    bkg_sigma = mad_std(frame)

    # We also need to guess the full-width at half maximum (FWHM) for the 
    # stars to guide our search, which we will verify in a later step.
    daofind = DAOStarFinder(fwhm=3, threshold=5.*bkg_sigma)
    sources = daofind(frame) 
    
    # Set up empty arrays to hold distances of detections from approximate locations
    if sources is None:
        numsources = 0
        print('No sources found in frame ',filename)
    else:
        numsources = len(sources)
    
    # Sometimes no sources or the wrong sources are found.  Ignore those.
    if numsources == 0:
        photresults[:,:,i] = np.nan # replace missing values with "not a number"
        fwhm.append(np.nan)
    else:
        
        tdist = np.zeros(numsources)
        cdist = np.zeros(numsources)
        # Populate arrays with distances for each source
        for ii,row in enumerate(sources):
            tdist[ii] = distance(tx,ty,row["x_centroid"],row["y_centroid"])
            cdist[ii] = distance(cx,cy,row["x_centroid"],row["y_centroid"])
    
        # "Typical background"
        bkgmed = np.median(frame.flatten())
        skybrightness.append(bkgmed)
    
        # Identify the row indices of the target and comp stars in "sources"
        tind = np.argmin(tdist)
        cind = np.argmin(cdist)
        # Replace "sources" with just target and comparison, in that order:
        sources = sources[[tind,cind]]

        x0 = sources['x_centroid'][0] # x center of target star
        y0 = sources['y_centroid'][0] # y center of target star
        dist = [] #empty list to hold pixel distances
        pval = [] #empty list to hold pixel values
        for x in np.arange(max([round(x0)-pdist,0]),min([round(x0)+pdist,frame.shape[0]])):
            for y in np.arange(max([round(y0)-pdist,0]),min([round(y0)+pdist,frame.shape[1]])):
                dist.append(distance(x,y,x0,y0))
                pval.append(frame[int(y),int(x)])

        p0=[20000.,1.] #initial guesses of height and sigma
        popt,_  = curve_fit(gaussian,dist,np.array(pval)-bkgmed,p0=p0, maxfev=10000)
        fwhm.append(np.abs(popt[-1])*2.3548*platescale)

        # Put star centroid positions in format for aperture photometry
        positions = [(x,y) for x,y in sources['x_centroid','y_centroid']]
        # Center aperture markers on centroids
        targetap.center = positions[0]
        targetan1.center = positions[0]
        targetan2.center = positions[0]
        compap.center = positions[1]
        compan1.center = positions[1]
        compan2.center = positions[1]

        # Loop through the different aperture sizes we want to try
        # and save the photometry.
        for j,apsize in enumerate(apsizes):
            # Define an example circular aperture of radius apsize pixels
            ap = CircularAperture(positions,r=apsize)
            ######## Define sky annulus in range 11-14 pixel radius#######
            skyan = CircularAnnulus(positions,r_in=20,r_out=24)
            # Do the photometry
            starapsum = ap.do_photometry(frame)[0]
            skysum = skyan.do_photometry(frame)[0]
            
            averagesky = skysum/skyan.area
            starsum = starapsum - ap.area*averagesky
            # Store the results!
            photresults[j,:,i] = starsum
            
        # If found centroid too far from expected, replace with np.nan
        if np.min(tdist) > 20:
            photresults[:,0,i] = np.nan
            fwhm[-1] = np.nan
        if np.min(cdist) > 20:
            photresults[:,1,i] = np.nan

    display(fig)
    clear_output(wait = True)
    plt.pause(0.0005)
           
    # Manually delete the image data to free up memory
    del frame
    gc.collect()

In [ ]:
for j, apsize in enumerate(apsizes):
    target = photresults[j,0,:]
    comp   = photresults[j,1,:]
print(target)

In [ ]:
plt.close()
header = fits.getheader(firstfilename)
header['EXPTIME']

#Now let's test apertures and pick the best light curve.

%matplotlib widget
exp_time = fits.getheader(firstfilename)['EXPTIME']

print(apsizes)
for j, apsize in enumerate(apsizes):
    target = photresults[j,0,:]
    comp   = photresults[j,1,:]

    # create a list of numbers to represent time for x axis
    # this list will be multiplied by 10 for sample data because every image is a 10 second exposure
    t0 = exp_time * np.arange(len(target))

    # create a divided light curve
    lc_div0 = target/comp

    lc_div = target/comp

    # mask out any nan values
    mask1 = np.isfinite(lc_div0) 
    lc_div = lc_div0[mask1]

    #normalize masked light curve
    lc_div_norm = lc_div/np.mean(lc_div)

    # apply mask to time as well 
    t1 = t0[mask1]
    
    ###### change the values of mask2 if you need to mask out outlier values #########
    mask2 = (lc_div_norm < 1.2) & (lc_div_norm > 0.87)
    lc_div_norm2 = lc_div_norm[mask2]/np.mean(lc_div_norm[mask2]) # mask light curve again and normalize
 
    # apply the second mask to the time as well
    t2 = t1[mask2]

    plt.figure(figsize = (12,3))
    plt.plot(t2, lc_div_norm2)
    plt.title(f'Aperture size: {apsize}')
    plt.show()

In [ ]:
plt.close()

# plot of fwhm througout the durations of the observing run 

fig = plt.figure()
ax1 = plt.subplot()

ax1.plot(t0,fwhm)

ax1.set_xlabel('time (sec)')
ax1.set_ylabel('FWHM (arcsec)')

###### change the file name for your target and comparison star #########

fig.savefig('7_24_26_R_Aql_comparison_seeing.pdf')

In [ ]:
# Based off of the light curves and the fwhm, pick the best light curve and save it.

###### Change the apsize here and within photresults ######

apsize = 7
target = photresults[7,0,:]
comp   = photresults[7,1,:]

 # create a list of numbers to represent time for x axis
    # this list will be multiplied by 10 for sample data because every image is a 10 second exposure
t0 = exp_time * np.arange(len(target))

    # create a divided light curve
lc_div0 = target/comp

lc_div = target/comp

    # mask out any nan values
mask1 = np.isfinite(lc_div0) 
lc_div = lc_div0[mask1]

    #normalize masked light curve
lc_div_norm = lc_div/np.mean(lc_div)

    # apply mask to time as well 
t1 = t0[mask1]
       
    ###### change the values of mask2 if you need to mask out outlier values #########
mask2 = (lc_div_norm < 1.2) & (lc_div_norm > 0.87)
lc_div_norm2 = lc_div_norm[mask2]/np.mean(lc_div_norm[mask2]) # mask light curve again and normalize
 
    # apply the second mask to the time as well
t2 = t1[mask2]

    

plt.figure(figsize = (12,3))
plt.plot(t2, lc_div_norm2)
plt.title(f'Aperture size: {apsize}')
plt.show()


###### change the file name for your target and comparison star#########

np.savetxt('7_24_26_R_Aql_comparison_light_curve.txt',out_arr,header=head)

#### Step Four
Since we need to repeat the aperture photometry, save the target photometry results.

In [ ]:
plt.close()
###### change the file name for your target and comparison star #########
#cleaned_target = target[~np.isnan(target)]
#np.save('7_24_26_R_Aql_target_photometry.npy', target)

#After you repeat, save the comparison data:
np.save('7_24_26_R_Aql_comparison_photometry.npy', target)

#### Step Five
Now, we have to do the same thing for our true comparison star. Pick your comparison star by finding a star in the field of view that has a known magnitude. AAVSO variable star plotter (https://apps.aavso.org/vsp/) may be helpful. Repeat Steps Three and Four with your comparison star.

#### Step Six
Now, we can calculate the magnitude based on the Zero-Point Flux of our comparison star.

In [ ]:
#First, we calculate the zero-point flux of the camparsion star.

#Known magnitude from V and B-V, found in the AAVSO VSP Photometry Chart
###### find your own values for B_V and V ######
B_V=0.860
V=12.333
B= B_V + V
print(B)

###### change the file name #########
counts = np.load('7_24_26_R_Aql_comparison_photometry.npy')
print(counts)

###### change the expousre time ######


ZP = V + 2.5*np.log10(counts/exp_time)
print(ZP)

In [ ]:
# Now we can calculate the magnitude of our target star based off of ZP.

###### change the file name #########
target = np.load('7_24_26_R_Aql_target_photometry.npy')

new_mag = []
for i in range(len(target)):
    zero = ZP[i]
    mag = zero - 2.5*np.log10(target[i]/exp_time)
    new_mag.append(mag)
print(new_mag)

###### change the file name #########
np.save('7_24_26_R_Aql_magnitude.npy', new_mag)

#### Step Seven
Let's make our own light curve based off the average magnitude from each night. Now, since Miras have very long periods, not much change is expected over a few weeks.

In [ ]:
# This will average the magnitudes, keep the dates, and create a new dataframe
mag_files = sorted(glob('/raid6/users/rantoine/Mira_Notebooks/Part_Three/*R_Aql_magnitude.npy'))

dates = []
avg_mag = []
for filepath in mag_files:
    filename = os.path.basename(filepath)
    date_str = filename.split('_R_Aql')[0] #taking the date from the filename
    date_time = datetime.strptime(date_str, "%m_%d_%y")
    #date = Time(date_time, scale="utc")
    
    mag = np.load(filepath)
    
    avg_mag.append(np.mean(mag))
    dates.append(date_time)
print(avg_mag)
print(dates)

df = pd.DataFrame({'Date': pd.to_datetime(dates), 'Avg_Mag': avg_mag})

In [ ]:
# Now, let's plot our lightcurve!
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 18
})

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(df['Date'], df['Avg_Mag'], color='#9C179E', s=10)
ax.set_xlabel('Time (Date)')
ax.set_ylabel('Magnitude (B)')
ax.invert_yaxis()
ax.set_title('Yerkes Light Curve of R Aql')

#Let's add a linear fit to verify. This part is generated by claude
x_numeric = mdates.date2num(df['Date'])
y_numeric = df['Avg_Mag'].values

slope, intercept = np.polyfit(x_numeric, y_numeric, 1)

print(f"Slope (mag per day): {slope:.6f}")

x_fit_dates = df['Date']
y_fit = slope * x_numeric + intercept
ax.plot(x_fit_dates, y_fit, color='black', alpha= 0.25, linestyle='--', label=f'Linear (Slope: {slope:.4f})')
ax.legend()
fig.autofmt_xdate()

In [ ]:
# Let's check if this matches up with the expected magnitude change
max_mag = 12
min_mag = 5.5
mag_range = max_mag - min_mag
bright = 0.4
dim = 0.6
period = 270
rate = mag_range/(period*bright)
print(rate)

***This is the end of Part Three! Move on to Part Four to bring it all together!***

In [ ]:
#comparing with gaia photometry
df['JD'] = Time(df['Date']).jd

gaia = Table.read('/raid6/users/rantoine/R_Aql/Light_Curves/EPOCH_PHOTOMETRY-Gaia DR3 4307450193249412352.fits')
gaia_time = gaia['g_transit_time']
gaia_mag = gaia['g_transit_mag']

bp_rp = gaia['bp_mag'] - gaia['rp_mag']
G = gaia['g_transit_mag']
V = -(-0.02704 + 0.01424*(bp_rp) - 0.215*(bp_rp**2) + 0.01426*(bp_rp**3) - G)
gaia['V_mag'] = V
#print(gaia['V_mag']) to check

#Now we need everything in JD, not TCB or HJD
#We will use astopy.time and start with Gaia
tcb_jd_corrected = gaia['g_transit_time'] + 2455197.5 #offset provided by gaia
gaia_tcb = Time(tcb_jd_corrected, scale='tcb', format = 'jd')
gaia_jd = gaia_tcb.jd
gaia['JD'] = gaia_jd
gaia.write("EPOCH_PHOTOMETRY-Gaia DR3 4307450193249412352.fits", overwrite=True)

#asassn = Table.read('R_Aql_asas-sn.fits')
#asassn_time = asassn['hjd']
#asassn_mag = asassn['mag']
#Note: To check yout column names, you can print(data.colnames) where data is your read .fits file.
print(gaia.colnames)
#Okay now lets set up our subplots.
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 10))

axes[0].scatter(df['JD'], df['Avg_Mag'], color='#9C179E', s=10)
axes[0].set_xlabel('Time (JD)')
axes[0].set_ylabel('Magnitude (B)')
axes[0].invert_yaxis()
axes[0].set_title('Yerkes Light Curve of R Aql')
axes[1].scatter(gaia_jd, V, color='red', s=10)
axes[1].set_xlabel('Time (JD)')
axes[1].set_ylabel('Magnitude (Gaia G)')
axes[1].set_title('Gaia DR3 Light Curve of R Aql')
axes[1].invert_yaxis()


axes[1].tick_params(axis='x', rotation=45)

axes[1].ticklabel_format(useOffset=False, style='plain', axis='x')

axes[0].tick_params(axis='x', rotation=45)

axes[0].ticklabel_format(useOffset=False, style='plain', axis='x')

def jd_to_year(jd):
    jd = np.atleast_1d(jd).astype(float)
    return Time(jd, format='jd').jyear

def year_to_jd(year):
    year = np.atleast_1d(year).astype(float)
    return Time(year, format='jyear').jd

secax = axes[0].secondary_xaxis('top', functions=(jd_to_year, year_to_jd))
secax.set_xlabel('Time (Year)')

secax = axes[1].secondary_xaxis('top', functions=(jd_to_year, year_to_jd))
secax.set_xlabel('Time (Year)')

plt.tight_layout()
plt.show()